<img src="https://www.luxonis.com/logo.svg" width="400">

# Conversion of PyTorch Model

## 🌟 Overview
In this tutorial, we'll go through converting a pre-trained PyTorch model. We'll first download the model, test its inference, and export it to ONNX format. We'll then make it ready for deployment on a Luxonis device and finally test it on a device.

## 📜 Table of Contents
- [🛠️ Installation](#installation)
- [🗃️ Model Download](#model-download)
- [✍ Model Test](#model-test)
- [📦 NN Archive](#nn-archive)
- [🗂️ Export and Archive](#export-and-archive)
- [🤖 Deploy](#deploy)
- [📷 DepthAI Script](#depthai-script)
- [🗂️ Export without Archive (Optional)](#onnx-export)

<a name="installation"></a>

## 🛠️ Installation

The main focus of this tutorial is using [`ModelConverter`](https://github.com/luxonis/modelconverter) for conversion of a pre-trained model [`ResNet-18`](https://pytorch.org/vision/stable/models/generated/torchvision.models.quantization.resnet18.html#resnet18) from `torchvision` to formats supported by Luxonis devices. `ModelConverter` is our open-source tool that supports conversion to all RVC Compiled Formats. Furthermore, we'll also use [`LuxonisML`](https://github.com/luxonis/luxonis-ml) since it provides us with functionality to generate a [`NN Archive`](https://rvc4.docs.luxonis.com/software/ai-inference/nn-archive/). Finally, we will use [`DepthAI v3`](https://rvc4.docs.luxonis.com/software/) and [`DepthAI Nodes`](https://rvc4.docs.luxonis.com/software/ai-inference/depthai-nodes/) to run the converted model, process and visualize the results. So, let's not wait any longer and get straight to it!

Install the required dependencies. This notebook pins the PyTorch/TorchVision pair validated for the ONNX export path.

In [ ]:
%pip install -q torch==2.10.0 torchvision==0.25.0 luxonis-ml==0.8.5 onnx==1.21.0 onnxruntime==1.23.2 depthai==3.7.1 depthai-nodes==0.5.1 modelconv==0.5.5
%pip install -q numpy==2.0.2

<a name="model-download"></a>

## 🗃️ Model Download

First, let's download the model from `torchvision`.

In [ ]:
import torchvision

# Load the pretrained ResNet-18 model
model = torchvision.models.resnet18(weights="IMAGENET1K_V1")
model.eval()  # Set the model to evaluation mode

<a name="model-test"></a>

## ✍ Model Test

It's a good practice to verify the performance of a source model that we want to convert to know that the model is working. This way, when the model is exported and isn't performing well on a device, we know that the problem must lie in the conversion process.

We will test the inference of the model on an image of a cat from a public dataset called [`crawford/cat-dataset`](https://www.kaggle.com/datasets/crawford/cat-dataset) available on Kaggle.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
from IPython.display import Image, display

media_dir = Path("media")
media_dir.mkdir(parents=True, exist_ok=True)

assets = {
    "cat.jpg": "https://raw.githubusercontent.com/luxonis/ai-tutorials/main/conversion/media/cat.jpg",
    "imagenet-simple-labels.json": "https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels/master/imagenet-simple-labels.json",
}

for filename, url in assets.items():
    path = media_dir / filename
    if not path.exists():
        urlretrieve(url, path)

img_file = media_dir / "cat.jpg"
labels_file = media_dir / "imagenet-simple-labels.json"

display(Image(filename=str(img_file)))

In [ ]:
import json
from PIL import Image
import torch

# Load the image
image = Image.open(img_file).convert("RGB")

# Define the transformation
transforms = torchvision.models.ResNet18_Weights.IMAGENET1K_V1.transforms(antialias=True)

# Preprocess the image
input_tensor = transforms(image).unsqueeze(0)

# Perform inference
with torch.no_grad():
    output = model(input_tensor)
    probabilities = torch.nn.functional.softmax(output[0], dim=0)

# Get the top-5 predictions
_, indices = torch.topk(probabilities, 5)

# Load the labels
with open(labels_file, "r") as f:
    labels = json.load(f)

print("Top-5 predictions:")
for i, idx in enumerate(indices, start=1):
    print(f"{i}) {labels[idx]}: {probabilities[idx].item()*100:.2f}%")

We have verified that the model returns reasonable predictions, so let's jump into the conversion.

<a name="nn-archive"></a>

## 📦 NN Archive

This section will introduce [`NN Archive`](https://rvc4.docs.luxonis.com/software/ai-inference/nn-archive/), what it is, and its benefit. `NN Archive` is our own format that packages the model executable(s) and configuration files into a .tar.xz archive. The primary purpose of the `NN Archive` is to describe and specify what the model expects as an input, what the model outputs, and lastly, if and how to process the result. The benefit of the `NN Archive` is seamless integration with our library ecosystem, especially the `DepthAI Nodes` package responsible for processing a model's output. Later in this tutorial, we will experience the benefit of this ourselves.

We will use functions from [`LuxonisML`](https://github.com/luxonis/luxonis-ml) to create the `NN Archive`. The `NN Archive` consists of two parts, model executables (e.g. `ONNX`, `OpenVINO IR`, `TFLite`) and a config encoding the scheme version and a dictionary describing a model's inputs, outputs, heads, and metadata sections. Let's briefly describe each section.

**Inputs**

This section describes all of the model's input(s) and their preprocessing. It's defined as a list of dictionaries. To check out all its fields, please visit our [documentation](https://rvc4.docs.luxonis.com/software/ai-inference/nn-archive/#NN%20Archive-Configuration-Inputs).

**Outputs**

This section specifies all the model's output(s). It's defined as a list of dictionaries containing the name and data type of the output data. For more information, refer to our [documentation](https://rvc4.docs.luxonis.com/software/ai-inference/nn-archive/#NN%20Archive-Configuration-Outputs).

**Head**

This section configures the post-processing steps applied to the model's output(s). It's again defined as a list of dictionaries. Please visit our [documentation](https://rvc4.docs.luxonis.com/software/ai-inference/nn-archive/#NN%20Archive-Configuration-Heads) to learn more about it.

**Metadata**

This section specifies the name of the model, the path to it, and the model's precision.


The creation of a `NN Archive` looks like this:

```python
from luxonis_ml.nn_archive.archive_generator import ArchiveGenerator
from luxonis_ml.nn_archive.config import CONFIG_VERSION


config = {
    "config_version": CONFIG_VERSION,       # Draw config version from luxonis-ml
    "model": {
        "metadata": { ... },                # Specify the model's metadata
        "inputs":   [ { ... }, ... ],       # Specify the model's input stream(s)
        "outputs":  [ { ... }, ... ],       # Specify the model's output stream(s)
        "heads":    [ { ... }, ... ],       # Specify all heads for the model
 }
}

generator = ArchiveGenerator(
    archive_name="...",                     # Name of the generated archive file
    save_path="...",                        # Path to the
    cfg_dict=config,
    executables_paths=["..."]
)

generator.make_archive()                    # Archive file is saved to the specified save_path
```

<a name="export-and-archive"></a>

## 🗂️ Export and Archive

Once we are satisfied with the model's performance, we want to prepare it for deployment on the device. This preparation consists of 2 steps. First, we want to export the model trained with PyTorch to a more general format called [`Open Neural Network Exchange (ONNX)`](https://onnx.ai/). Then, we want to package this exported model into a `NN Archive` as described in the section above.

Let's start by exporting the model to ONNX.

In [ ]:
import torch

onnx_model_path = "resnet18.onnx"
input_tensor = torch.randn(1, 3, 224, 224)  # Random input tensor

torch.onnx.export(
    model,
    input_tensor,
    onnx_model_path,
    input_names=["images"],
    output_names=["output"],
    opset_version=18,
    dynamo=False,  # Use legacy exporter for RVC2/OpenVINO MO compatibility.
)

The code below creates the `NN Archive`.

In [ ]:
from luxonis_ml.nn_archive import ArchiveGenerator
from luxonis_ml.nn_archive.config_building_blocks import (
    DataType,
    InputType,
)
from luxonis_ml.nn_archive.config import CONFIG_VERSION

# Define the configuration dictionary
config = {
    "config_version": CONFIG_VERSION, # draw config version from luxonis-ml
    "model": {
        "metadata": {
            "name": "resnet18",
            "path": "resnet18.onnx",
            "precision": DataType.FLOAT32
        },
        "inputs": [ # Specify all inputs to the model
            {
                "name": "images",  # Define the input tensor name
                "dtype": DataType.FLOAT32, # Define the input tensor data type
                "input_type": InputType.IMAGE,
                "shape": [1, 3, 224, 224], # Define the input tensor shape
                "layout": "NCHW", # Define the input tensor order
                "preprocessing": {
                    "mean": [123.675, 116.28, 103.53], # Mean values for each channel applied during preprocessing
                    "scale": [58.395, 57.12, 57.375], # Scale values for each channel applied during preprocessing
                    "dai_type": "RGB888p" # Data type for DepthAI
                }
            }
        ],
        "outputs": [  # Specify all outputs from the model
            {
                "name": "output", # Define the output tensor name
                "dtype": DataType.FLOAT32, # Define the output tensor data type
                "shape": [1, 1000] # Define the output tensor shape
            }
        ],
        "heads": [ # Specify all heads for the model
            {
                "parser": "ClassificationParser", # Define the parser to use from depthai-nodes
                "metadata": {
                    "postprocessor_path": None,
                    "is_softmax": False, # Whether the model output is softmaxed
                    "n_classes": 1000, # Number of classes in the model
                    "classes": labels # List of class labels
                },
                "outputs": ["output"] # Define the output tensor to use for the head
            }
        ]
    }
}

archive = ArchiveGenerator(
    archive_name="resnet18", # Define string name of the generated archive.
    save_path="./", # Define string path to where you want to save the archive file.
    cfg_dict=config,
    executables_paths=["resnet18.onnx"], # Define a list of string paths to relevant model executables.
)
archive.make_archive()

<a name="deploy"></a>

## 🤖 Deploy

Now that we have successfully archived the model, we aim to deploy it to the Luxonis device. The model's specific format depends on the Luxonis device series you have. We will show you how to use our [`HubAI SDK`](https://github.com/luxonis/hubai-sdk) to convert the model as simply as possible.


We will use the `HubAI SDK` Python API, which leverages our [`HubAI`](https://hub.luxonis.com) platform to perform model conversion in the background. To get started, you'll need to create an account on `HubAI` and obtain your team’s API key.

We'll start by installing the `HubAI SDK` package.

In [ ]:
%pip install -q hubai-sdk==0.3.0

We can store the API key in a variable. You can get your API key from the [HubAI website](https://hub.luxonis.com/team-settings).

In [ ]:
import os
import getpass
from hubai_sdk import HubAIClient

if not os.environ.get("HUBAI_API_KEY"):
    os.environ["HUBAI_API_KEY"] = getpass.getpass("Enter your HubAI API key: ")

Model conversion can be done via either the CLI or the Python API — here, we'll use the latter. For more information, see the [HubAI SDK documentation](https://github.com/luxonis/hubai-sdk).

The call below creates a new model card within your team on `HubAI`, uploads the model file and metadata, then performs cloud-side conversion to the selected target platform (e.g., [`RVC2`](https://rvc4.docs.luxonis.com/hardware/platform/rvc/rvc2/), [`RVC4`](https://rvc4.docs.luxonis.com/hardware/platform/rvc/rvc4/)). Once completed, the converted model is automatically downloaded to your device.

For HubAI-specific conversion parameters, refer to the [parameter documentation](https://github.com/luxonis/hubai-sdk/blob/main/docs/available_parameters.md#model-parameters). Platform-specific parameters are also documented there.


In [ ]:
from hubai_sdk import HubAIClient

client = HubAIClient(api_key=os.environ["HUBAI_API_KEY"])

ARTIFACT_DIR = Path("resnet18-224x224-exported-to-rvc2")
ARTIFACT_DIR.mkdir(exist_ok=True)
# =============================================================================
# RVC2 conversion
# =============================================================================
response = client.convert.RVC2(
    path="resnet18.tar.xz",
    name="Resnet18",
    description_short="Pretrained Resnet18 on ImageNet",
    tasks=["CLASSIFICATION"],
    license_type="MIT",
    is_public=False,
    output_dir=str(ARTIFACT_DIR),
)

MODEL_PATH = Path(response.downloaded_path)
print("Model artifact:", MODEL_PATH)

# Equivalent command using the CLI
# !hubai login
# !hubai convert rvc2 --path "resnet18.tar.xz" \
#                                 --name "Resnet18" \
#                                 --description-short "Pretrained Resnet18 on ImageNet" \
#                                 --tasks "CLASSIFICATION" \
#                                 --license-type "MIT" \
#                                 --no-is-public

# =============================================================================
# RVC4 conversion
# =============================================================================
# converted_model = convert(
#     "rvc4",
#     path="resnet18.tar.xz",
#     name="Resnet18",
#     description_short="Pretrained Resnet18 on ImageNet",
#     tasks=["CLASSIFICATION"],
#     license_type="MIT",
#     quantization_data="general",
#     is_public=False
# )

# Equivalent command using the CLI
# !hubai login
# !hubai convert rvc4 --path "resnet18.tar.xz" \
#                                 --name "Resnet18" \
#                                 --description-short "Pretrained Resnet18 on ImageNet" \
#                                 --tasks "classification" \
#                                 --license-type "MIT" \
#                                 --quantization-data "GENERAL" \
#                                 --no-is-public

We can see for ourselves that this call really created a new model card on `HubAI` with the exported model.

<img src="./media/resnet_model_exported.png" alt="Exported model on HubAI" width="800">

We have successfully converted our model for RVC2, so let's test it on the camera! Please copy the path to the downloaded archive with the converted model from the output log of the appropriate code cell; we will use it in the next section.

<a name="depthai-script"></a>

## 📷 DepthAI Script

To test our model on one of our cameras, we need to have `DepthAI v3` and `Depthai Nodes` installed. Moreover, the following script must be run locally and requires a Luxonis device connected to your machine.

To run the model on a DepthAI device using the script below, please note the following:

- You can view the output stream by opening [http://localhost:8082](http://localhost:8082) in your browser.

- If you're running the script from a Jupyter Notebook, the output may not appear directly within the notebook. The script should print a link pointing to [http://localhost:8082](http://localhost:8082) for accessing the stream.

- To stop the video stream, press **`q`** while focused on the visualizer page.



In [ ]:
from pathlib import Path

import depthai as dai
from depthai_nodes.node import ParsingNeuralNetwork
import gc
import time

DEVICE = None  # Set to None to use the default device, or specify a device IP/MXID.

# Use the RVC2 artifact downloaded from HubAI.
# Prefer this if the previous conversion cell defined `response`.
if "response" in globals():
    MODEL_PATH = Path(response.downloaded_path)
else:
    # Or hardcode it only if needed:
    MODEL_PATH = Path("resnet18-224x224-exported-to-rvc2/resnet18.rvc2.tar.xz")

available_devices = dai.Device.getAllAvailableDevices()

if DEVICE is None and not available_devices:
    print("No DepthAI devices found.")
    print("Run this cell locally with an OAK / RVC2 Luxonis device connected.")
else:
    device = dai.Device(dai.DeviceInfo(DEVICE)) if DEVICE else dai.Device()
    platform = device.getPlatform()

    print("Connected device platform:", platform.name)
    print("Using model:", MODEL_PATH)

    img_frame_type = (
        dai.ImgFrame.Type.BGR888i
        if platform.name == "RVC4"
        else dai.ImgFrame.Type.BGR888p
    )

    visualizer = dai.RemoteConnection(httpPort=8082)

    with dai.Pipeline(device) as pipeline:
        cam = pipeline.create(dai.node.Camera).build()
        nn_archive = dai.NNArchive(str(MODEL_PATH))

        # Create the neural network node
        nn_with_parser = pipeline.create(ParsingNeuralNetwork).build(
            cam.requestOutput((224, 224), type=img_frame_type, fps=30),
            nn_archive,
        )

        # Configure the visualizer node
        visualizer.addTopic(topicName="rgb", output=nn_with_parser.passthrough)
        visualizer.addTopic(topicName="classifications", output=nn_with_parser.out)

        # Start pipeline
        pipeline.start()
        visualizer.registerPipeline(pipeline)
        
        print("Open http://localhost:8082 in your browser.")
        print("Press q in the visualizer window to stop.")
        print("Avoid interrupting the notebook kernel while the pipeline is running.")

        try:
            while pipeline.isRunning():
                key = visualizer.waitKey(1)
                if key == ord("q"):
                    print("Stopping pipeline.")
                    break
        except KeyboardInterrupt:
            print("Interrupted by user. Stopping pipeline.")
        finally:
            pipeline.stop()

            # Jupyter keeps cell variables alive between runs. Release the
            # DepthAI objects so the visualizer ports are freed before rerunning.
            del visualizer
            del pipeline
            del device

            gc.collect()
            time.sleep(0.5)

            print("Pipeline stopped.")

<a name="onnx-export"></a>

## 🗂️ Export without Archive (Optional)

It is also possible to skip the model archiving and convert the model straight from `ONNX.` However, when running the model on the device, we'd need to define parsers and other parameters manually, so we recommend first creating the `NN Archive` and then converting the model.

In [ ]:
from hubai_sdk import HubAIClient
from pathlib import Path

ONNX_ARTIFACT_DIR = Path("resnet18-onnx-exported-to-rvc2")
ONNX_ARTIFACT_DIR.mkdir(exist_ok=True)

client = HubAIClient(api_key=os.environ["HUBAI_API_KEY"])

response = client.convert.RVC2(
    path="resnet18.onnx",
    name="Resnet18 ONNX",
    description_short="Pretrained Resnet18 on ImageNet",
    tasks=["CLASSIFICATION"],
    license_type="MIT",
    is_public=False,
    output_dir=str(ARTIFACT_DIR),
)

ONNX_MODEL_PATH = Path(response.downloaded_path)
print("Model artifact:", ONNX_MODEL_PATH)

Yay! 🎉🎉🎉 Huge congratulations, you have successfully finished this tutorial in which you deployed a pre-trained ResNet18 classification model to our cameras!